# HW14 — эмбеддинги, FAISS, оценка retrieval и mini-RAG

Домашняя работа к семинару 14: воспроизводимый pipeline по локальной базе знаний (регламент учебной лаборатории ML).

**Содержание:** загрузка документов → чанкинг → эмбеддинги → индекс FAISS → контрольные запросы и метрики → эксперимент по `top_k` → обновление базы и переиндексация → mini-RAG (extractive) → разбор ошибок и экспорт артефактов.

**Про скорость:** первый запуск качает веса модели с Hugging Face (долго только один раз). Дальше эмбеддинги чанков берутся из кэша в `artifacts/` — повторный **Run All** заметно быстрее. На CPU используется компактная модель `rubert-tiny2`.

**Кружок в шапке Cursor** чаще всего — индексация/анализ редактора, а не ваш код. Смотрите на **ноутбук**: у выполняемой ячейки слева крутится индикатор или стоит `[ * ]`. Функция `chunk_by_words` на этих данных работает доли секунды; долго обычно только ячейка **«эмбеддинги + FAISS»** (загрузка модели и первый подсчёт векторов).


In [ ]:
# При необходимости установите зависимости (раскомментируйте строку ниже и выполните эту ячейку).
# %pip — только внутри Jupyter; в PowerShell/терминале используйте: python -m pip install ...
# %pip install -q faiss-cpu sentence-transformers torch

import hashlib
import json
import os
import random
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch

    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("device:", DEVICE)


def hw14_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "homeworks" / "HW14", cwd.parent, cwd.parent / "homeworks" / "HW14"]
    for c in candidates:
        p = c / "data" / "kb.json"
        if p.is_file():
            return c
    raise FileNotFoundError("Не найден data/kb.json — запустите ноутбук из корня репозитория или из homeworks/HW14.")


HW14_ROOT = hw14_root()
ARTIFACTS = HW14_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("HW14_ROOT =", HW14_ROOT)


In [ ]:
def chunk_by_words(text: str, chunk_size: int = 55, overlap: int = 12) -> List[str]:
    words = text.replace("\n", " ").split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть > 0")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть < chunk_size")
    step = chunk_size - overlap
    out: List[str] = []
    for start in range(0, len(words), step):
        piece = words[start : start + chunk_size]
        if piece:
            out.append(" ".join(piece))
    return out


def documents_to_chunks(documents: List[Dict[str, str]], chunk_size: int, overlap: int) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for doc in documents:
        did, title, text = doc["doc_id"], doc["title"], doc["text"]
        for i, ch in enumerate(chunk_by_words(text, chunk_size=chunk_size, overlap=overlap)):
            rows.append(
                {
                    "doc_id": did,
                    "title": title,
                    "chunk_id": f"{did}__{i}",
                    "chunk_index": i,
                    "chunk_text": ch,
                }
            )
    return pd.DataFrame(rows)


def load_kb(path: Path) -> List[Dict[str, str]]:
    data = json.loads(path.read_text(encoding="utf-8"))
    for d in data:
        assert "doc_id" in d and "title" in d and "text" in d
    return data


KB_PATH = HW14_ROOT / "data" / "kb.json"
documents = load_kb(KB_PATH)
print("Число документов:", len(documents))
display(pd.DataFrame(documents).head(5))
display(
    Markdown(
        "**Предметная область:** внутренние правила учебной лаборатории ML (доступ, данные, GPU, сдача работ). "
        "По ним естественно задавать вопросы и искать отрывки — типичный сценарий retrieval и RAG."
    )
)


In [ ]:
example_doc = documents[5]
print("Пример документа:", example_doc["doc_id"], example_doc["title"])
chunks_preview = chunk_by_words(example_doc["text"], chunk_size=55, overlap=12)
for j, c in enumerate(chunks_preview[:3]):
    print(f"\n--- chunk {j} ({len(c.split())} слов) ---\n", c[:400], "...")
display(
    Markdown(
        "**Чанкинг:** разбиение по словам с перекрытием `overlap`, чтобы границы абзацев не «рвали» смысл. "
        "Параметры по умолчанию: `chunk_size=55`, `overlap=12`."
    )
)


In [ ]:
CHUNK_SIZE = 55
OVERLAP = 12
TOP_K_DEFAULT = 5

chunks_df = documents_to_chunks(documents, CHUNK_SIZE, OVERLAP)
print("Число чанков:", len(chunks_df))
chunks_df.head(3)


In [ ]:
from sentence_transformers import SentenceTransformer

print(
    "[HW14] Эта ячейка самая тяжёлая: сначала загрузка модели (при первом запуске — минуты + интернет), затем векторы чанков.",
    flush=True,
)

# Компактная русскоязычная модель — заметно быстрее MiniLM-L12 на CPU.
# Для сравнения можно сменить на: "paraphrase-multilingual-MiniLM-L12-v2" (точнее, но дольше).
MODEL_NAME = "cointegrated/rubert-tiny2"
ENCODE_BATCH = 64

print(f"[HW14] Качаю/инициализирую {MODEL_NAME!r}... (подождите, без вывода это норма)", flush=True)
embedder = SentenceTransformer(MODEL_NAME, device=DEVICE)
print("[HW14] Модель готова.", flush=True)


def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms


def encode_chunks(texts: List[str], *, show_bar: bool) -> np.ndarray:
    emb = embedder.encode(
        texts,
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        show_progress_bar=show_bar,
        normalize_embeddings=False,
    ).astype(np.float32)
    return l2_normalize(emb)


def texts_fingerprint(texts: Sequence[str]) -> str:
    h = hashlib.sha256(MODEL_NAME.encode("utf-8"))
    for t in texts:
        h.update(t.encode("utf-8", errors="replace"))
        h.update(b"\xff")
    return h.hexdigest()[:28]


def load_or_encode(texts: List[str], *, label: str = "chunks") -> np.ndarray:
    """Повторные прогоны: эмбеддинги из npz в artifacts/ (пока не менялись тексты/модель)."""
    fid = texts_fingerprint(texts)
    path = ARTIFACTS / f"emb_{label}_{fid}.npz"
    if path.is_file():
        print(f"[HW14] Эмбеддинги из кэша ({label}): {path.name}", flush=True)
        return np.load(path)["emb"].astype(np.float32)
    n = len(texts)
    print(
        f"[HW14] Считаем эмбеддинги ({label}), чанков: {n} (ниже может быть progress bar)...",
        flush=True,
    )
    emb = encode_chunks(texts, show_bar=n > 8)
    np.savez_compressed(path, emb=emb)
    return emb


chunk_texts = chunks_df["chunk_text"].tolist()
chunk_emb = load_or_encode(chunk_texts, label="base")
dim = chunk_emb.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(chunk_emb)

print("FAISS index ntotal:", index.ntotal, "dim:", dim)
assert index.ntotal == len(chunks_df), "Число векторов в FAISS должно совпадать с числом строк chunks_df"


def faiss_search(ix: faiss.Index, query: str, top_k: int) -> Tuple[np.ndarray, np.ndarray]:
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=False).astype(np.float32)
    q = np.ascontiguousarray(l2_normalize(q), dtype=np.float32)
    n = int(ix.ntotal)
    if n == 0:
        return np.array([], dtype=np.float32), np.array([], dtype=np.int64)
    k = min(int(top_k), n)
    scores, idx = ix.search(q, k)
    s = np.asarray(scores[0], dtype=np.float32)
    i = np.asarray(idx[0], dtype=np.int64)
    mask = (i >= 0) & (i < n)
    return s[mask], i[mask]


def search_index(query: str, top_k: int) -> Tuple[np.ndarray, np.ndarray]:
    return faiss_search(index, query, top_k)


demo_queries = [
    "Как получить доступ к GPU для нейросетей?",
    "Что делать с персональными данными в лаборатории?",
    "До какого времени можно работать в лаборатории по будням?",
]

for q in demo_queries:
    display(Markdown(f"### Запрос: {q}"))
    sc, ix = search_index(q, TOP_K_DEFAULT)
    display(chunks_df.iloc[ix][["doc_id", "title", "chunk_text"]].assign(score=sc))


In [ ]:
benchmark: List[Dict[str, str]] = [
    {"query": "До скольки открыта лаборатория и нужен ли пропуск после 18:00?", "expected_source": "mlab_02"},
    {"query": "Можно ли ставить любые программы на рабочие станции?", "expected_source": "mlab_03"},
    {"query": "Как поступить при утечке или инциденте с данными?", "expected_source": "mlab_04"},
    {"query": "Как оформить зависимости Python для сдачи работы?", "expected_source": "mlab_05"},
    {"query": "Нужна ли заявка для обучения нейросети на видеокарте?", "expected_source": "mlab_06"},
    {"query": "Зачем фиксировать seed в Jupyter и чистить вывод?", "expected_source": "mlab_07"},
    {"query": "Есть ли штраф за опоздание сдачи и сколько?", "expected_source": "mlab_08"},
    {"query": "Нужно ли проверять смещения выборки перед метриками?", "expected_source": "mlab_09"},
    {"query": "Как связаться с администратором по технике?", "expected_source": "mlab_15"},
    {"query": "Разрешён ли Docker и что запрещено при его использовании?", "expected_source": "mlab_13"},
]


def evaluate_retrieval(
    bench: Sequence[Dict[str, str]],
    chunks: pd.DataFrame,
    top_k: int,
) -> pd.DataFrame:
    rows = []
    for item in bench:
        q = item["query"]
        exp = item["expected_source"]
        scores, idx = search_index(q, top_k)
        retrieved = chunks.iloc[idx]
        doc_ids = retrieved["doc_id"].tolist()
        hit = int(exp in doc_ids)
        ranks = [r + 1 for r, d in enumerate(doc_ids) if d == exp]
        rank_first = int(min(ranks)) if ranks else np.nan
        n_rel = int((chunks["doc_id"] == exp).sum())
        in_top = int((retrieved["doc_id"] == exp).sum())
        recall_k = float(in_top / n_rel) if n_rel else 0.0
        rows.append(
            {
                "query": q,
                "expected_source": exp,
                "retrieved_sources": "|".join(doc_ids),
                "hit_at_k": hit,
                "rank_of_first_relevant": rank_first,
                "recall_at_k": recall_k,
            }
        )
    return pd.DataFrame(rows)


eval_df = evaluate_retrieval(benchmark, chunks_df, TOP_K_DEFAULT)
hit_rate = eval_df["hit_at_k"].mean()
recall_rate = eval_df["recall_at_k"].mean()
print(f"hit@{TOP_K_DEFAULT} =", hit_rate)
print(f"recall@{TOP_K_DEFAULT} (доля чанков нужного документа в top-k) =", recall_rate)
display(eval_df)

eval_df.to_csv(ARTIFACTS / "retrieval_eval.csv", index=False, encoding="utf-8")
print("saved", ARTIFACTS / "retrieval_eval.csv")


In [ ]:
# Эксперимент: сравниваем два значения top_k (5 vs 12) при тех же чанках и модели.

for tk in [5, 12]:
    df = evaluate_retrieval(benchmark, chunks_df, tk)
    print(f"top_k={tk}  hit@k={df['hit_at_k'].mean():.3f}  mean recall@k={df['recall_at_k'].mean():.3f}")

display(
    Markdown(
        "**Вывод:** большее `top_k` обычно повышает `hit@k`, но может подтягивать шум в контекст RAG; для основной части оставляем `TOP_K_DEFAULT`."
    )
)

fig, ax = plt.subplots(figsize=(6, 3))
xs = [5, 12]
hits = [evaluate_retrieval(benchmark, chunks_df, k)["hit_at_k"].mean() for k in xs]
ax.bar([str(x) for x in xs], hits, color=["#4C72B0", "#55A868"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("hit@k")
ax.set_title("Сравнение top_k на одном benchmark")
plt.tight_layout()
plt.show()


## Сравнительный эксперимент по параметрам retrieval

Проверяем, как `chunk_size`, `overlap` и `top_k` влияют на качество поиска.  
Для каждой конфигурации заново строим чанки → эмбеддинги → FAISS → оцениваем на том же benchmark.

In [ ]:
# Сравнительный эксперимент: chunk_size / overlap / top_k
# Оценивается hit@k и recall@k на том же benchmark (10 запросов).
# Эмбеддинги кэшируются в artifacts/ — повторный запуск быстрый.

RETRIEVAL_CONFIGS = [
    {"chunk_size": 30,  "overlap": 5,  "top_k": 3},
    {"chunk_size": 30,  "overlap": 5,  "top_k": 5},
    {"chunk_size": 55,  "overlap": 12, "top_k": 3},   # ← baseline (DEFAULT)
    {"chunk_size": 55,  "overlap": 12, "top_k": 5},
    {"chunk_size": 55,  "overlap": 12, "top_k": 10},
    {"chunk_size": 80,  "overlap": 20, "top_k": 5},
    {"chunk_size": 80,  "overlap": 20, "top_k": 10},
    {"chunk_size": 120, "overlap": 30, "top_k": 5},
]


def eval_config(cfg: Dict[str, int], docs: List[Dict[str, str]]) -> Dict[str, Any]:
    cs, ov, tk = cfg["chunk_size"], cfg["overlap"], cfg["top_k"]
    cdf = documents_to_chunks(docs, chunk_size=cs, overlap=ov)
    label = f"cs{cs}_ov{ov}"
    emb = load_or_encode(cdf["chunk_text"].tolist(), label=label)
    ix = faiss.IndexFlatIP(emb.shape[1])
    ix.add(emb)

    rows = []
    for item in benchmark:
        q, exp = item["query"], item["expected_source"]
        _, idx = faiss_search(ix, q, tk)
        doc_ids = cdf.iloc[idx]["doc_id"].tolist()
        hit = int(exp in doc_ids)
        n_rel = int((cdf["doc_id"] == exp).sum())
        in_top = int(sum(1 for d in doc_ids if d == exp))
        recall_k = float(in_top / n_rel) if n_rel else 0.0
        rows.append({"hit": hit, "recall": recall_k})

    hit_rate = float(np.mean([r["hit"] for r in rows]))
    recall_rate = float(np.mean([r["recall"] for r in rows]))
    return {
        "chunk_size": cs,
        "overlap": ov,
        "top_k": tk,
        "n_chunks": len(cdf),
        f"hit@k": round(hit_rate, 3),
        f"recall@k": round(recall_rate, 3),
    }


# Базовые документы (до обновления) для честного сравнения
# documents_before сохранён выше; если ячейка запущена отдельно — используем documents[:len(documents)-3]
_docs_for_exp = documents_before if "documents_before" in dir() else documents[:len(documents)-3]

exp_rows = []
for cfg in RETRIEVAL_CONFIGS:
    result = eval_config(cfg, _docs_for_exp)
    exp_rows.append(result)
    flag = " ← baseline" if cfg["chunk_size"] == 55 and cfg["overlap"] == 12 and cfg["top_k"] == 3 else ""
    print(
        f"chunk_size={cfg['chunk_size']:3d}  overlap={cfg['overlap']:2d}  top_k={cfg['top_k']:2d}"
        f"  →  hit@k={result['hit@k']:.3f}  recall@k={result['recall@k']:.3f}"
        f"  (чанков: {result['n_chunks']}){flag}"
    )

exp_df = pd.DataFrame(exp_rows)
exp_df.to_csv(ARTIFACTS / "retrieval_param_experiment.csv", index=False, encoding="utf-8")
display(exp_df)
print("saved", ARTIFACTS / "retrieval_param_experiment.csv")


In [ ]:
# Визуализация результатов эксперимента

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

labels = [f"cs{r['chunk_size']}/ov{r['overlap']}/k{r['top_k']}" for _, r in exp_df.iterrows()]
x = np.arange(len(labels))
width = 0.6

for ax, metric, color, title in [
    (axes[0], "hit@k",    "#4C72B0", "hit@k по конфигурациям"),
    (axes[1], "recall@k", "#55A868", "recall@k по конфигурациям"),
]:
    bars = ax.bar(x, exp_df[metric], width=width, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel(metric)
    ax.set_title(title)
    # baseline (индекс 2 в RETRIEVAL_CONFIGS)
    ax.axhline(exp_df[metric].iloc[2], color="red", linewidth=1, linestyle="--", label="baseline")
    ax.legend(fontsize=8)
    for bar, val in zip(bars, exp_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7)

plt.suptitle("Сравнение конфигураций retrieval (chunk_size / overlap / top_k)", fontsize=11)
plt.tight_layout()
plt.show()

display(Markdown(
    "**Вывод эксперимента:**  \n"
    "- Увеличение `top_k` при фиксированных `chunk_size`/`overlap` надёжно повышает `hit@k` и `recall@k`, "
    "но добавляет шум в контекст RAG.  \n"
    "- Мелкие чанки (`chunk_size=30`) хуже при малом `top_k`: один факт часто разрезан, "
    "а релевантный чанк не попадает в топ.  \n"
    "- Крупные чанки (`chunk_size=120`) уменьшают число чанков и снижают точность совпадения при `top_k=5`.  \n"
    "- Оптимальный компромисс на этих данных: `chunk_size=55`, `overlap=12`, `top_k≥5`."
))


In [ ]:
NEW_DOCUMENTS: List[Dict[str, str]] = [
    {
        "doc_id": "mlab_16",
        "title": "Очередь GPU и бронирование слотов",
        "text": "С 2026 года введена электронная очередь на GPU-сервер lab-gpu-01. Слоты по 30 минут бронируются в календаре с названием учебной группы. Одновременно не более двух учебных заданий на одной карте. Если задача падает с OOM, уменьшите batch size и перезапустите в следующем слоте. При нарушении расписания доступ может быть ограничён на неделю.",
    },
    {
        "doc_id": "mlab_17",
        "title": "Политика fair-share на GPU",
        "text": "Планировщик fair-share снижает приоритет длинных задач, чтобы короткие эксперименты не простаивали. Приоритет повышается, если в логе видно учебный тег курса. Фоновые задачи без тега переводятся на ночные окна 23:00–07:00. Мониторинг загрузки доступен на внутренней странице статуса кластера.",
    },
    {
        "doc_id": "mlab_18",
        "title": "Что такое OOM и как его избежать",
        "text": "OOM означает нехватку видеопамяти. Уменьшите размер батча, используйте смешанную точность там, где это разрешено заданием, или укоротите последовательности. После OOM процесс нужно завершить, чтобы освободить память для других студентов. В отчёте укажите финальные гиперпараметры, при которых обучение стабильно.",
    },
]

queries_for_update = [
    "Как бронировать слот на GPU-сервере и что будет при нарушении расписания?",
    "Как планировщик fair-share влияет на длинные задачи?",
    "Что означает OOM на GPU и какие первые шаги предпринять?",
]

documents_before = list(documents)
chunks_before = chunks_df.copy()
index_before = index


def build_artifacts(docs: List[Dict[str, str]], chunk_size: int, overlap: int):
    cdf = documents_to_chunks(docs, chunk_size, overlap)
    emb = load_or_encode(cdf["chunk_text"].tolist(), label="updated")
    ix = faiss.IndexFlatIP(emb.shape[1])
    ix.add(emb)
    assert ix.ntotal == len(cdf)
    return cdf, emb, ix


documents_after = documents_before + NEW_DOCUMENTS
chunks_after, emb_after, index_after = build_artifacts(documents_after, CHUNK_SIZE, OVERLAP)

rows_ba = []
for q in queries_for_update:
    _, idx_b = faiss_search(index_before, q, TOP_K_DEFAULT)
    b_src = "|".join(chunks_before.iloc[idx_b]["doc_id"].tolist())

    _, idx_a = faiss_search(index_after, q, TOP_K_DEFAULT)
    a_src = "|".join(chunks_after.iloc[idx_a]["doc_id"].tolist())

    changed = int(b_src != a_src)
    rows_ba.append(
        {
            "query": q,
            "before_retrieved_sources": b_src,
            "after_retrieved_sources": a_src,
            "changed": changed,
        }
    )

# основной пайплайн переводим на обновлённую базу
documents = documents_after
chunks_df = chunks_after
chunk_emb = emb_after
index = index_after

before_after_df = pd.DataFrame(rows_ba)
display(before_after_df)
before_after_df.to_csv(ARTIFACTS / "retrieval_before_after_update.csv", index=False, encoding="utf-8")
print("saved", ARTIFACTS / "retrieval_before_after_update.csv")
print("Документов после обновления:", len(documents), "чанков:", len(chunks_df))


In [ ]:
def split_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]


def generate_answer_from_context(query: str, context_lines: List[str], max_sentences: int = 2) -> str:
    pool: List[str] = []
    for line in context_lines:
        pool.extend(split_sentences(line))
    pool = [s for s in pool if len(s.split()) >= 4]
    if not pool:
        return "Недостаточно контекста для ответа."
    vec = TfidfVectorizer(ngram_range=(1, 2))
    mat = vec.fit_transform([query] + pool).toarray().astype(np.float32)
    qv, pv = mat[0], mat[1:]
    qn = np.linalg.norm(qv) + 1e-12
    pn = np.linalg.norm(pv, axis=1) + 1e-12
    scores = (pv @ qv) / (pn * qn)
    order = np.argsort(-scores)
    picked: List[str] = []
    seen = set()
    for i in order:
        if scores[i] <= 0:
            continue
        s = pool[i]
        key = s.lower().strip()
        if key in seen:
            continue
        seen.add(key)
        picked.append(s)
        if len(picked) >= max_sentences:
            break
    if not picked:
        return "В контексте нет уверенно релевантного фрагмента."
    return " ".join(picked)


def mini_rag(question: str, top_k: int = TOP_K_DEFAULT) -> Dict[str, Any]:
    scores, idx = search_index(question, top_k)
    retrieved = chunks_df.iloc[idx].copy().reset_index(drop=True)
    retrieved.insert(0, "rank", np.arange(1, len(retrieved) + 1))
    retrieved["score"] = scores
    context_lines = []
    sources: List[str] = []
    for _, r in retrieved.iterrows():
        tag = f"[{r['doc_id']}] {r['title']}"
        sources.append(tag)
        context_lines.append(r["chunk_text"])
    answer = generate_answer_from_context(question, context_lines, max_sentences=2)
    return {
        "question": question,
        "answer": answer,
        "retrieved_sources": "|".join(sources),
        "retrieved": retrieved,
    }


rag_questions = [
    "Можно ли коммитить большие датасеты в git и что использовать вместо этого?",
    "Какие правила для удалённого SSH-доступа через VPN?",
    "Что делать, если модель падает с OOM на GPU?",
    "Как сдавать работы после дедлайна и какой штраф?",
    "Что запрещено при использовании Docker в лаборатории?",
]

rag_rows = []
for rq in rag_questions:
    out = mini_rag(rq, top_k=TOP_K_DEFAULT)
    display(Markdown(f"**Вопрос:** {rq}\n\n**Ответ:** {out['answer']}\n\n**Источники:** {out['retrieved_sources']}"))
    rag_rows.append(
        {
            "question": out["question"],
            "answer": out["answer"],
            "retrieved_sources": out["retrieved_sources"],
        }
    )

rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv(ARTIFACTS / "rag_examples.csv", index=False, encoding="utf-8")
print("saved", ARTIFACTS / "rag_examples.csv")


In [ ]:
display(
    Markdown(
        "## Анализ ошибок и ограничений\n\n"
        "1. **Extractive-ответчик** опирается на лексическое сходство предложений с вопросом; если ответ сформулирован иными словами, чем запрос, качество падает (см. семинар S14-demo-03).\n"
        "2. **Слишком общий запрос** может поднять релевантный, но расплывчатый чанк из другого регламента.\n"
        "3. **Границы чанков:** факт может быть разрезан; без overlap или с малым `top_k` контекст неполный.\n"
        "4. **Обновление базы:** новые документы конкурируют по сходству; иногда верхний чанк — близкий по теме, но не лучший по сути ответа.\n"
    )
)
